# Dual-Stream Deepfake Detector — Colab Training Driver
Run cells top to bottom. Runtime > Change runtime type > GPU (T4) first.

In [ ]:
!git clone https://github.com/Vishak05/dual-stream-deepfake-detection.git
%cd dual-stream-deepfake-detection
!pip install -e .[colab]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/data'  # or a path under /content/drive/MyDrive for persistence

In [ ]:
import json, os

# Kaggle's UI shows your API key as a copyable string, not a downloadable file.
# Kaggle > Settings > API > "Create New Token" to get the key; your username
# is shown on the same page / in your profile URL (kaggle.com/<username>).
KAGGLE_USERNAME = "your-kaggle-username"
KAGGLE_KEY = "paste-your-copied-api-key-here"

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!python scripts/download_datasets.py \
  --kaggle-dataset xhlulu/140k-real-and-fake-faces \
  --out-dir $DATA_DIR

## StyleGAN3 synthetic augmentation
Clone NVlabs/stylegan3 and download a pretrained checkpoint (see README) before running the generation cell below — **or**, for a first-pass smoke test, run the placeholder cell instead and come back to real generation later.

In [ ]:
!mkdir -p $DATA_DIR/stylegan3
# Placeholder for a first smoke-test run. Skip the real generation cell below
# for now; come back and run it later once you have a checkpoint, then rerun
# the manifest-building cell so augmented_manifest.csv picks up real images.

In [ ]:
!git clone https://github.com/NVlabs/stylegan3.git
!python scripts/generate_stylegan3.py \
  --stylegan3-repo stylegan3 \
  --checkpoint /content/drive/MyDrive/checkpoints/stylegan3-r-ffhq-1024x1024.pkl \
  --out-dir $DATA_DIR/stylegan3 \
  --num-images 5000

**Before training:** check your actual CPU core count with `!nproc` and set `--num-workers`
to match (Colab's free tier is often only 2 cores — passing more workers than cores can
make loading *slower*, not faster). For a first pass, add `--epochs 2` to both commands
below as a quick smoke test before committing to the full 20-epoch run; if a run gets
interrupted, just rerun the same command with the same `--out-dir` — it automatically
resumes from the last completed epoch.

## Build manifests, then train baseline and dual-stream models
Build `original_manifest.csv`, `augmented_manifest.csv`, and `test_manifest.csv` from the downloaded folders using `dsfnet.data.manifest` (see README 'Building manifests'), then train:

In [ ]:
!find $DATA_DIR -maxdepth 4 -type d | sort

!python scripts/train_model.py --manifest $DATA_DIR/original_manifest.csv --model spatial_only --out-dir runs/baseline --num-workers 2
!python scripts/train_model.py --manifest $DATA_DIR/augmented_manifest.csv --model dual_stream --out-dir runs/dual_stream --num-workers 2

In [ ]:
from pathlib import Path
from dsfnet.data.manifest import (
    build_original_manifest, build_stylegan3_augmented_manifest,
    scan_source_folder, write_manifest_csv,
)

DATASET_ROOT = Path(DATA_DIR) / 'real_vs_fake' / 'real-vs-fake'

original = build_original_manifest(
    real_sources={'Real': DATASET_ROOT / 'train' / 'real'},
    fake_sources={'StyleGAN': DATASET_ROOT / 'train' / 'fake'},
)
write_manifest_csv(original, Path(DATA_DIR) / 'original_manifest.csv')
print(f'original_manifest.csv: {len(original)} entries')

augmented = build_stylegan3_augmented_manifest(original, Path(DATA_DIR) / 'stylegan3')
write_manifest_csv(augmented, Path(DATA_DIR) / 'augmented_manifest.csv')
print(f'augmented_manifest.csv: {len(augmented)} entries')

test_entries = (
    scan_source_folder(DATASET_ROOT / 'test' / 'real', label=0, source='Real')
    + scan_source_folder(DATASET_ROOT / 'test' / 'fake', label=1, source='StyleGAN')
)
write_manifest_csv(test_entries, Path(DATA_DIR) / 'test_manifest.csv')
print(f'test_manifest.csv: {len(test_entries)} entries')

In [ ]:
!python scripts/train_model.py --manifest $DATA_DIR/original_manifest.csv --model spatial_only --out-dir runs/baseline
!python scripts/train_model.py --manifest $DATA_DIR/augmented_manifest.csv --model dual_stream --out-dir runs/dual_stream

In [ ]:
!python scripts/evaluate_models.py \
  --test-manifest $DATA_DIR/test_manifest.csv \
  --baseline-checkpoint runs/baseline/spatial_only.pth --baseline-model spatial_only \
  --candidate-checkpoint runs/dual_stream/dual_stream.pth --candidate-model dual_stream \
  --out-report runs/fpr_comparison_report.json